<b>Transform Customer Data

1. Remove records with NULL customer_id

2. Remove exact duplicate records

3. Remove duplicate records based on created_timestamp

4. Cast column values to the correct data types

5. Write the transformed data to the Silver schema

In [0]:
df = spark.sql("""
            SELECT *
            FROM gizmobox.bronze.v_customers
""")
display(df)

In [0]:
df = spark.table('gizmobox.bronze.py_customers')
display(df)

In [0]:
df = spark.read.table('gizmobox.bronze.py_customers')
display(df)

<b>1. Remove records with NULL customer_id



In [0]:
df = spark.table('gizmobox.bronze.py_customers')
df_filtered = df.filter('customer_id is not null')
display (df_filtered)

In [0]:
df_filtered = df.filter(df.customer_id.isNotNull())
display (df_filtered)

In [0]:
df_filtered = df.where(df.customer_id.isNotNull())
display (df_filtered)

<b>2. Remove exact duplicate records



In [0]:
df_distinct = df_filtered.distinct()
display(df_distinct)

In [0]:
df_distinct = df_filtered.dropDuplicates()
display(df_distinct)

<b>3. Remove duplicate records based on created_timestamp



In [0]:
from pyspark.sql import functions as F

df_max_ts = df_distinct.groupBy('customer_id') \
                        .agg(F.max('created_timestamp').alias('max_created_timestamp'))

display(df_max_ts)
    

In [0]:
df_distinct_customer = (
                        df_distinct
                        .join(df_max_ts, (df_distinct.customer_id == df_max_ts.customer_id) & 
                              (df_distinct.created_timestamp == df_max_ts.max_created_timestamp), 'inner')
                        .select(df_distinct['*'])
)
display(df_distinct_customer)

<b>4. Cast the column values to the correct data type










In [0]:
df_casted_customer = (
    df_distinct_customer
        .select(
            df_distinct_customer.created_timestamp.cast('timestamp'),
            df_distinct_customer.customer_id,
            df_distinct_customer.customer_name,
            df_distinct_customer.date_of_birth.cast('date'),
            df_distinct_customer.email,
            df_distinct_customer.member_since.cast('date'),
            df_distinct_customer.telephone
        )
)

display(df_casted_customer)

<b> 5. Write Data to a Delata Table 

In [0]:
df_casted_customer.writeTo('gizmobox.silver.py_customers').createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_customers